In [ ]:
import torch
import kaolin as kal
from kaolin.visualize import IpyTurntableVisualizer
from kaolin.render.camera import Camera

# Suppose your model is something like:
#    class MyRenderer(torch.nn.Module):
#         def forward(self, poses, times):  # poses: batch of extrinsics + intrinsics; times optional
#             # returns a torch.FloatTensor, shape (B, 3, H, W), values in [0,1]
#             ...

H, W = 64, 114

model = MyRenderer().cuda()
model.eval()

# Define the camera / default pose

# Create a Kaolin Camera object -- you need to understand how to map your pose format
# to what kaolin Camera expects: intrinsics, extrinsics.

# Example camera initialization (you will adapt intrinsics etc)
# Suppose focal, principal point etc; or use Kaolin's camera utility
cam = Camera(fov=60.0,  # field of view or whatever
             resolution=(H, W),
             # maybe you need near, far, etc depending on your model
             )

# Now write a render function that Kaolin can call:
def render_fn(kal_camera: Camera) -> torch.ByteTensor:
    """
    This will be called by the visualizer whenever the camera moves etc.
    """

    # 1. Get the camera_pose (extrinsic) + intrinsics + maybe time if your model uses it
    # This depends on how your model wants the pose format.
    # Suppose model takes a tensor of shape (1, ...pose_params...) + a time scalar
    
    # Extract pose from kal_camera
    # Example: you might do something like (this is pseudocode, adapt to your format):
    pose = kal_camera.get_world_to_camera_matrix()   # or something like that
    intrinsics = kal_camera.get_intrinsics()        # focal, principal, etc.

    # Possibly package them into your model's expected format
    # Suppose you concat them to form camera_poses_and_frame_time
    frame_time = torch.tensor([0.0], device='cuda')  # or whatever you want
    # Build input
    camera_input = build_camera_input_from_pose_intrinsics(pose, intrinsics, frame_time)
    # Pass through model
    with torch.no_grad():
        image_tensor = model(camera_input)  # shape e.g. (1, 3, H, W), on GPU

    # Move to CPU, convert to uint8, permute channels
    # Suppose image_tensor ∈ [0,1]
    img = image_tensor.clamp(0.0, 1.0)[0]  # remove batch dim: shape (3, H, W)
    img = (img * 255.0).to(torch.uint8)
    img = img.permute(1, 2, 0).cpu()  # now (H, W, 3), ByteTensor

    return img

# Optionally, make a fast_render_fn for lower resolution or fewer samples etc.

# Now create the visualizer
viz = IpyTurntableVisualizer(
    height=H,
    width=W,
    camera=cam,
    render=render_fn,
    # fast_render=fast_render_fn,  # optional
    focus_at=torch.zeros(3),        # maybe center of scene
    world_up_axis=1,                 # depends on your coordinate system
    max_fps=15                       # adjust if slow
)

# Show in notebook
viz.show()

In [ ]:
import kaolin
import math

In [ ]:
from threedgrut_playground.engine import Engine3DGRUT
from threedgrut_playground.utils.composition import join_gaussians

# Start the engine with the env Gaussian object.
# The engine will set the scene scale and rescale added meshes according to the scene size
engine = Engine3DGRUT(
    gs_object=gs_env,
    mesh_assets_folder=mesh_assets_folder,
    envmap_assets_folder=envmap_assets_folder,
    default_config=default_config
)

# Load another 3dgrt Gaussian object, and translate up along z axis to position above table
# Then concatenate both Gaussians into a single 3dgrut object the engine can render
with torch.no_grad():
    env_mog = engine.scene_mog
    object_mog,_ = engine.load_3dgrt_object(gs_object, config_name=default_config)
    object_mog.positions[:,2] = object_mog.positions[:,2] + 1.3
    engine.scene_mog = join_gaussians(env_mog, object_mog)

In [ ]:
from threedgrut_playground.engine import OptixPrimitiveTypes

# The engine is loaded with a sample mesh primitive (glass sphere),
# remove the default mesh primitives from the engine here
for mesh_name in list(engine.primitives.objects.keys()):
    engine.primitives.remove_primitive(mesh_name)

# Instead, we'll add another mesh object to a scene.
# For now, we load and display it with a simple Lambertian shader.
# To respect the original look, we'll later display as a Physically Based Rendered (PBR) mesh.
print(f'Available meshes: {list(engine.primitives.assets.keys())}')
engine.primitives.add_primitive(
    geometry_type='Spray_bottle',
    primitive_type=OptixPrimitiveTypes.DIFFUSE, # Lambertian
    device='cuda'
)

# Take note of the mesh name, we'll need it to reference this primitive through the engine
mesh_name = list(engine.primitives.objects.keys())[0]
# Reference the mesh object, as OptixPrimitive, to quickly access the geometry and transforms later
prim = engine.primitives.objects[mesh_name]

# Position the mesh nicely within the scene, i.e. above the table
engine.primitives.objects[mesh_name].transform.tx += 0.95
engine.primitives.objects[mesh_name].transform.tz += 1.5
engine.primitives.objects[mesh_name].transform.rz += 25.0
engine.primitives.objects[mesh_name].transform.ry += 20.0

# Configure some initial rendering settings - to low and fast quality
engine.camera_type = 'Pinhole'
engine.camera_fov = 60.0
engine.use_spp = False                # Disable antialiasing for now
engine.antialiasing_mode = '4x MSAA'  # Set the default antialiasing to 4x MSAA, if enabled later
engine.use_optix_denoiser = False     # Disable Optix denoiser

# Set a HDR envmap as background, to provide some light to our mesh
engine.environment.set_env('drackenstein_quarry_puresky_4k.hdr')
engine.environment.ibl_intensity=0.60
engine.environment.exposure=-0.04
engine.environment.envmap_offset = [-0.5, -0.25]

# Finally, let engine know it should refresh internal structures due to changes
engine.invalidate_materials_on_gpu()  # Sync newly loaded materials (loaded with the mesh)
engine.rebuild_bvh(engine.scene_mog)                 # Rebuild gaussians BVH
engine.primitives.rebuild_bvh_if_needed(True, True)  # Rebuild meshes BVH

In [ ]:
def set_fast_quality_mode():
    engine.use_spp = False
    engine.antialiasing_mode = '4x MSAA'
    engine.spp.mode = 'msaa'
    engine.spp.spp = 4
    engine.spp.reset_accumulation()
    engine.use_optix_denoiser = False

    # Use simple Lambertian shading for all mesh objects
    for prim in engine.primitives.objects.values():
         prim.primitive_type = OptixPrimitiveTypes(OptixPrimitiveTypes.DIFFUSE)

def set_medium_quality_mode():
    engine.use_spp = True
    engine.antialiasing_mode = '8x MSAA'
    engine.spp.mode = 'msaa'
    engine.spp.spp = 4
    engine.spp.reset_accumulation()
    engine.use_optix_denoiser = True

    # Use Path Tracing for all mesh objects
    for prim in engine.primitives.objects.values():
         prim.primitive_type = OptixPrimitiveTypes(OptixPrimitiveTypes.PBR)

def set_high_quality_mode():
    # Make sure engine renders high quality frames
    engine.use_spp = True
    engine.antialiasing_mode = 'Quasi-Random (Sobol)'
    engine.spp.mode = 'low_discrepancy_seq'
    engine.spp.spp = 64
    engine.spp.reset_accumulation()
    engine.use_optix_denoiser = True

    # Use Path Tracing for all mesh objects
    for prim in engine.primitives.objects.values():
         prim.primitive_type = OptixPrimitiveTypes(OptixPrimitiveTypes.PBR)

In [ ]:
from PIL import Image

# Render a frame from the engine
camera = kaolin.render.easy_render.default_camera(512)
renderbuffer = engine.render(camera)

# Convert to PIL so Jupyter can display it
rgb_buffer = renderbuffer['rgb']
rgb_buffer = (rgb_buffer[0] * 255).to(torch.uint8)
image = Image.fromarray(rgb_buffer.cpu().numpy())

display(image)

In [ ]:
# Start by setting up a kaolin camera object --

def create_camera(H, W):
    return Camera.from_args(
        eye=torch.ones((3,)), at=torch.zeros((3,)), up=torch.tensor([0., 1., 0.]),
        fov=math.pi * 45 / 180, height=H, width=W
    )

def reset_camera(cam):
    # Position the Kaolin camera to observe the table
    cam.update(torch.tensor(
        [[-0.51,   0.86,  0.00,  0.00],
         [-0.25,  -0.14,  0.96,  0.02],
         [ 0.83,   0.49,  0.29, -7.31],
         [ 0.00,   0.00,  0.00,  1.00]],
        dtype=cam.dtype, device=cam.device
    ))


H, W = 64, 114

# Create initial camera
camera = create_camera(H, W)

# Set it to Blender coordinates (Z axis pointing upwards)
camera.change_coordinate_system(
    torch.tensor([[1, 0, 0],
                  [0, 0, 1],
                  [0, -1, 0]]
))
camera = camera.cuda()

# ..and position above objects of interest
reset_camera(camera)

In [ ]:
def fast_render(in_cam, **kwargs):
    # Called during interactions, disables effects for quick rendering
    framebuffer = engine.render_pass(in_cam, is_first_pass=True)
    # Alpha channel is the opacity output map from the renderer
    if engine.environment.is_ignore_envmap():
        alpha = framebuffer['opacity']
    else: # However if using an env map - alpha channel is always 1, since we render the background
        alpha = torch.ones_like(framebuffer['opacity'])
    # Read back the rendered rgb buffer and convert to a format supported by the visualizer
    rgba_buffer = torch.cat([framebuffer['rgb'], alpha], dim=-1)
    rgba_buffer = torch.clamp(rgba_buffer, 0.0, 1.0)
    return (rgba_buffer[0] * 255).to(torch.uint8) # Convert to RGBA image of uint8 pixels of [0,255]


def render(in_cam, **kwargs):
    # Called when the user stops interacting, to generate a high quality frame
    is_first_pass = engine.is_dirty(in_cam)
    framebuffer = engine.render_pass(in_cam, is_first_pass=True)
    # Note: this loop will stall until all passes are done
    while engine.has_progressive_effects_to_render():
        framebuffer = engine.render_pass(in_cam, is_first_pass=False)
    # Alpha channel is the opacity output map from the renderer
    if engine.environment.is_ignore_envmap():
        alpha = framebuffer['opacity']
    else: # However if using an env map - alpha channel is always 1, since we render the background
        alpha = torch.ones_like(framebuffer['opacity'])
    # Read back the rendered rgb buffer and convert to a format supported by the visualizer
    rgba_buffer = torch.cat([framebuffer['rgb'], alpha], dim=-1)
    rgba_buffer = torch.clamp(rgba_buffer, 0.0, 1.0)
    return (rgba_buffer[0] * 255).to(torch.uint8) # Convert to RGBA image of uint8 pixels of [0,255]


# Initialize the visualizer with the 2 render functions above, and a camera object.
visualizer = kaolin.visualize.IpyTurntableVisualizer(
    height=camera.height,
    width=camera.width,
    camera=camera,
    render=render,
    fast_render=fast_render,
    max_fps=8,
    world_up_axis=2
)
# without a GUI, showing the interactive renderer is as simple as:
# visualizer.show()

# But we want to include some widgets.. and therefore we'll include some extra code below